# VascuMap Batch Pipeline

Two workflow modes:

1. **GUI Curation** — Launch `CurationApp` to review & curate device ROI + organoid mask
   for every image in a napari viewer, then run the full pipeline unattended.
2. **Headless Automatic** — Skip the GUI and run fully automatic device segmentation +
   pipeline on every image.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

from liffile import LifFile

from vascumap import VascuMap
from gui_region_detection import CurationApp
from models import Pix2Pix, load_segmentation_model

W0416 20:15:16.151000 26456 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff image files in source_dir and return (source, files, jobs)."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")
    image_files = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff"))

    def determine_mask_mode(file_path, image_name=""):
        """Infer dark/light/False mask mode from file and image name keywords."""
        if force_mask is not None:
            return force_mask
        combined = (file_path.name + " " + image_name).lower()
        if "marina" in combined and "bead" in combined:
            return "light"
        return "dark" if "marina" in combined else False

    jobs = []
    for file_path in image_files:
        if file_path.suffix.lower() == ".lif":
            try:
                with LifFile(file_path) as lif:
                    for idx, image in enumerate(lif.images):
                        image_name = getattr(image, "name", "")
                        if require_merged and "merged" not in image_name.lower():
                            continue
                        jobs.append((file_path, idx, determine_mask_mode(file_path, image_name), image_name))
            except Exception as exc:
                print(f"[SKIP] {file_path.name}: {exc}")
        else:
            if require_merged and "merged" not in file_path.name.lower():
                continue
            jobs.append((file_path, 0, determine_mask_mode(file_path), file_path.stem))
    return source, image_files, jobs


def run_batch_from_curation(curated_jobs, output_base, save_all_interim=False,
                            model_p2p=None, model_unet=None, skip_names=None):
    """Run the full pipeline on curated jobs whose status is 'curated'."""
    results = []
    if skip_names is None:
        skip_names = set()
    Path(output_base).mkdir(parents=True, exist_ok=True)
    processable = [j for j in curated_jobs if j.status == "curated"
                   and hasattr(j, 'finalised_outputs') and j.finalised_outputs is not None]
    for i, job in enumerate(processable, 1):
        name = job.image_name
        if name in skip_names:
            print(f"\n[{i}/{len(processable)}] {name}  — SKIP (already perfect)")
            results.append((name, "SKIP_PERFECT", ""))
            continue
        print(f"\n{'='*70}\n[{i}/{len(processable)}] {name}\n{'='*70}")
        try:
            vm = VascuMap(curated_outputs=job.finalised_outputs, model_p2p=model_p2p, model_unet=model_unet)
            output_dir = Path(output_base) / vm.image_name
            print(f"  Output → {output_dir}")
            vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
            results.append((vm.image_name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((name, "FAILED", str(exc)))
    succeeded = sum(1 for _, s, _ in results if s == "OK")
    skipped_perfect = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    skipped_curation = sum(1 for j in curated_jobs if j.status == "skip")
    print(f"\n{'='*70}\nBatch complete: {succeeded}/{len(results)} succeeded, "
          f"{skipped_perfect} skipped (perfect), {skipped_curation} skipped (curation)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    return results


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              start_index=1, skip_names=None):
    """Run all jobs in headless automatic mode (no GUI curation)."""
    results = []
    if skip_names is None:
        skip_names = set()
    Path(output_base).mkdir(parents=True, exist_ok=True)
    failure_diag_dir = Path(output_base) / "FAILED_diagnostics"
    for i, (file_path, idx, mask_flag, image_name) in enumerate(jobs, start_index):
        lif_tag = f" (LIF #{idx}: {image_name})" if file_path.suffix.lower() == ".lif" else ""
        if file_path.suffix.lower() == ".lif":
            safe_name = image_name.replace("/", "_").replace("\\", "_")
            expected_name = f"{file_path.stem}_{safe_name}_img{idx}"
        else:
            expected_name = file_path.stem
        if expected_name in skip_names:
            print(f"\n[{i}/{len(jobs)}] {file_path.name}{lif_tag}  — SKIP (already perfect)")
            results.append((expected_name, "SKIP_PERFECT", ""))
            continue
        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {file_path.name}{lif_tag}  mask={mask_flag}\n{'='*70}")
        try:
            vm = VascuMap(
                image_source_path=str(file_path), image_index=idx,
                device_width_um=device_width_um, mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel, model_p2p=model_p2p,
                model_unet=model_unet, failure_output_dir=str(failure_diag_dir),
            )
            vm.image_name = expected_name
            output_dir = Path(output_base) / vm.image_name
            print(f"  Output → {output_dir}")
            vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
            results.append((vm.image_name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((file_path.name + lif_tag, "FAILED", str(exc)))
    succeeded = sum(1 for _, s, _ in results if s == "OK")
    skipped_perfect = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    print(f"\n{'='*70}\nBatch complete: {succeeded}/{len(results)} succeeded, {skipped_perfect} skipped (perfect)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"Z:\Maria Cuende\0_Projects\0_Placenta\0_Barrier4PE\3D system\Exp4\Vascumap"
output_base = r"Z:\Bel\Individual_Vascumap_Outputs\Maria_Vascumap"

use_gui               = True    # True = napari curation, False = fully automatic
device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = False
save_all_interim      = False

In [5]:
# ── Discover jobs ──────────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(source_dir, force_mask, require_merged)
print(f"Source: {source}\nFiles: {len(image_files)}  |  Jobs: {len(jobs)}")
for i, (file_path, idx, mask, image_name) in enumerate(jobs, 1):
    lif_tag = f" (LIF image {idx}: '{image_name}')" if file_path.suffix.lower() == ".lif" else ""
    print(f"  {i}. {file_path.name}{lif_tag}  mask={mask}")

# Skip images already categorised as perfect
perfect_dir = Path(output_base)
if perfect_dir.is_dir():
    perfect_names = {d.name for d in perfect_dir.iterdir() if d.is_dir()}
    print(f"Found {len(perfect_names)} already-processed images in {perfect_dir}")
else:
    print(f"[WARN] Output directory not found: {perfect_dir}")
    perfect_names = set()

Source: Z:\Maria Cuende\0_Projects\0_Placenta\0_Barrier4PE\3D system\Exp4\Vascumap
Files: 1  |  Jobs: 24
  1. 20260415_Exp4_PE.lif (LIF image 0: 'P1_1')  mask=False
  2. 20260415_Exp4_PE.lif (LIF image 1: 'P2-8')  mask=False
  3. 20260415_Exp4_PE.lif (LIF image 2: 'P1-2')  mask=False
  4. 20260415_Exp4_PE.lif (LIF image 3: 'P1-4')  mask=False
  5. 20260415_Exp4_PE.lif (LIF image 4: 'P5-1')  mask=False
  6. 20260415_Exp4_PE.lif (LIF image 5: 'P5-2')  mask=False
  7. 20260415_Exp4_PE.lif (LIF image 6: 'P5-3')  mask=False
  8. 20260415_Exp4_PE.lif (LIF image 7: 'P5-4')  mask=False
  9. 20260415_Exp4_PE.lif (LIF image 8: 'P5-6')  mask=False
  10. 20260415_Exp4_PE.lif (LIF image 9: 'P2-1')  mask=False
  11. 20260415_Exp4_PE.lif (LIF image 10: 'P2-2')  mask=False
  12. 20260415_Exp4_PE.lif (LIF image 11: 'P2-3')  mask=False
  13. 20260415_Exp4_PE.lif (LIF image 12: 'P2-4')  mask=False
  14. 20260415_Exp4_PE.lif (LIF image 13: 'P2-6')  mask=False
  15. 20260415_Exp4_PE.lif (LIF image 14: 'P3_

In [ ]:
# ── GUI curation (run this cell, curate in napari, then run next cell) ─────────
if use_gui:
    app = CurationApp(jobs, device_width_um=device_width_um)
    app.open()
    # Napari viewer is now open.
    # Navigate: n = next, b = back, a = accept, s = skip
    # Edit the device ROI (red polygon) and organoid mask (labels layer).
    # When done curating all images, run the next cell.
else:
    print("use_gui is False — skip this cell and run the next one for headless mode.")

Auto-detecting device regions for 24 images...
  [1/24] P1_1 ...   [LIF] Raw array shape: (48, 6526, 3773)  dtype=uint8
  [LIF] Final stack shape: (48, 6526, 3773)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
OK
  [2/24] P2-8 ... 

In [ ]:
# ── Finalise curation + run batch pipeline ─────────────────────────────────────
if use_gui:
    curated_jobs = app.finalise()
    run_batch_from_curation(curated_jobs, output_base, save_all_interim,
                            shared_model_p2p, shared_model_unet,
                            skip_names=perfect_names)
else:
    run_batch(jobs, output_base, device_width_um, brightfield_channel,
              save_all_interim, shared_model_p2p, shared_model_unet,
              skip_names=perfect_names)